In [ ]:
import pandas as pd
import numpy as np
from dash import Dash, dcc, html, dash_table, Input, Output
import plotly.express as px
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Load Excel file
file_path = "ab.xlsx"
df = pd.read_excel(file_path)

df.columns = df.columns.str.strip()

if "PART TIME" in df.columns:
    df["PART TIME"] = df["PART TIME"].map({True: "Yes", False: "No"})
if "EXTRACURRICULAR" in df.columns:
    df["EXTRACURRICULAR"] = df["EXTRACURRICULAR"].map({True: "Yes", False: "No"})

subject_columns = ["MATHS", "HISTORY", "PHYSICS", "CHEMISTRY", "BIOLOGY", "ENGLISH", "GEOGRAPHY"]
available_subjects = [col for col in subject_columns if col in df.columns]

def evaluate_performance(row):
    threshold = 50
    return "Pass" if np.mean([row[subj] for subj in available_subjects]) >= threshold else "Fail"

df["ACTUAL PASS/FAIL"] = df.apply(evaluate_performance, axis=1)

# S3PSO Implementation
class S3PSO:
    def __init__(self, num_particles=10, max_iter=50):
        self.num_particles = num_particles
        self.max_iter = max_iter
        self.global_best = None
        self.global_best_score = float('-inf')

    def fitness(self, weights):
        preds = df[available_subjects].dot(weights) / np.sum(weights)
        preds = np.where(preds >= 50, "Pass", "Fail")
        return accuracy_score(df["ACTUAL PASS/FAIL"], preds)

    def optimize(self):
        particles = np.random.rand(self.num_particles, len(available_subjects))
        velocities = np.random.rand(self.num_particles, len(available_subjects))
        personal_best = particles.copy()
        personal_best_scores = np.array([self.fitness(p) for p in particles])

        for i in range(self.max_iter):
            for j in range(self.num_particles):
                score = self.fitness(particles[j])
                if score > personal_best_scores[j]:
                    personal_best_scores[j] = score
                    personal_best[j] = particles[j]
                if score > self.global_best_score:
                    self.global_best_score = score
                    self.global_best = particles[j]
                inertia = 0.5 + np.random.rand() / 2
                cognitive = 1.5 * np.random.rand() * (personal_best[j] - particles[j])
                social = 1.5 * np.random.rand() * (self.global_best - particles[j])
                velocities[j] = inertia * velocities[j] + cognitive + social
                particles[j] += velocities[j]
        return self.global_best

s3pso = S3PSO()
best_weights = s3pso.optimize()
df["PREDICTED PASS/FAIL"] = np.where(df[available_subjects].dot(best_weights) / np.sum(best_weights) >= 50, "Pass", "Fail")

precision = precision_score(df["ACTUAL PASS/FAIL"], df["PREDICTED PASS/FAIL"], pos_label="Pass")
recall = recall_score(df["ACTUAL PASS/FAIL"], df["PREDICTED PASS/FAIL"], pos_label="Pass")
f1 = f1_score(df["ACTUAL PASS/FAIL"], df["PREDICTED PASS/FAIL"], pos_label="Pass")
accuracy = accuracy_score(df["ACTUAL PASS/FAIL"], df["PREDICTED PASS/FAIL"])

# Grouping for subject-wise graph
students_per_group = 10
total_students = len(df)
group_count = (total_students + students_per_group - 1) // students_per_group

def get_subject_fig(group_index):
    start_idx = group_index * students_per_group
    end_idx = start_idx + students_per_group
    df_subset = df.iloc[start_idx:end_idx]
    df_subjects = df_subset.melt(id_vars=["FIRST NAME"], value_vars=available_subjects)
    return px.bar(
        df_subjects, x="FIRST NAME", y="value", color="variable",
        title=f"Subject-wise Marks (Group {group_index + 1})",
        labels={"value": "Marks", "variable": "Subjects"}
    )

fig_extracurricular = px.pie(df, names="EXTRACURRICULAR", title="Extracurricular Activities Participation") if "EXTRACURRICULAR" in df.columns else None
fig_absence = px.bar(df, x="FIRST NAME", y="ABSENCE DAYS", title="Absence Days per Student", color="ABSENCE DAYS") if "ABSENCE DAYS" in df.columns else None

# Dashboard
app = Dash(__name__)
app.layout = html.Div([
    html.H1("Student Dashboard", style={'textAlign': 'center', 'color': 'blue'}),
    dash_table.DataTable(
        id="student-table",
        data=df.to_dict('records'),
        columns=[{"name": col, "id": col} for col in df.columns],
        page_size=10,
        row_selectable="single",
        style_table={'overflowX': 'auto'}
    ),
    html.Div(id="student-details", style={'fontSize': '18px', 'marginTop': '20px', 'color': 'darkblue'}),

    html.Br(),
    html.H3("Select Group to View Subject-wise Marks"),
    dcc.Dropdown(
        id='group-dropdown',
        options=[{'label': f'Group {i+1}', 'value': i} for i in range(group_count)],
        value=0,
        clearable=False
    ),
    dcc.Graph(id='subject-graph'),

    html.Br(),
    html.H3("Absence Days Analysis"),
    dcc.Graph(figure=fig_absence) if fig_absence else html.P("No absence data available."),

    html.Br(),
    html.H3("Extracurricular Activities"),
    dcc.Graph(figure=fig_extracurricular) if fig_extracurricular else html.P("No extracurricular data available."),

    html.Br(),
    html.H3("S3PSO Prediction Performance"),
    dash_table.DataTable(
        data=[
            {"Method": "S3PSO", "Precision": precision, "Recall": recall, "F1-Measure": f1, "Accuracy": accuracy}
        ],
        columns=[{"name": col, "id": col} for col in ["Method", "Precision", "Recall", "F1-Measure", "Accuracy"]],
        style_table={'overflowX': 'auto'}
    ),
    html.Br(),
    dcc.Graph(
        figure=px.bar(
            x=["Precision", "Recall", "F1-Measure", "Accuracy"],
            y=[precision, recall, f1, accuracy],
            title="S3PSO Performance Metrics",
            labels={"x": "Metric", "y": "Score"},
            color_discrete_sequence=['green']
        )
    )
])

@app.callback(
    Output("student-details", "children"),
    Input("student-table", "selected_rows")
)
def display_student_details(selected_rows):
    if not selected_rows:
        return "Click on a student to see details."
    student_data = df.iloc[selected_rows[0]].to_dict()
    return html.Div([html.Div([html.B(f"{key}: "), f"{value}"]) for key, value in student_data.items()])

@app.callback(
    Output("subject-graph", "figure"),
    Input("group-dropdown", "value")
)
def update_subject_graph(group_index):
    return get_subject_fig(group_index)

if __name__ == '__main__':
    app.run(debug=True)



In [ ]:
!pip install dash
